# 🎮 Sysadmin Game: GRPO Training with Live Environment

This notebook trains a Qwen2.5-Coder model using **Group Relative Policy Optimization (GRPO)** with live Docker container feedback.

**Key difference from SFT**: Instead of learning from static traces, the model interacts with real broken Linux containers and learns from actual fix/fail outcomes.

**Requirements**: 
- GPU runtime (T4 minimum, A100 recommended)
- Docker support (we'll set this up)

---

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Docker in Colab
%%capture
!apt-get update
!apt-get install -y docker.io
!service docker start
!docker --version

In [ ]:
# Verify Docker is running
!docker info --format '{{.ServerVersion}}' || echo 'Docker not running, trying again...'
!service docker start
!sleep 2
!docker info --format '{{.ServerVersion}}'

In [ ]:
# Install Unsloth (optimized for Colab)
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# Install other dependencies
%%capture
!pip install trl>=0.8.0 transformers>=4.40.0 datasets>=2.18.0 wandb>=0.16.0 matplotlib peft bitsandbytes docker

In [ ]:
# Clone the sysadmin-game repository
!git clone https://github.com/shettydevesh/sysadmin.git
%cd sysadmin

In [ ]:
# Build the systemd-enabled sandbox image
!docker build -t sysadmin-sandbox:latest -f docker/sandbox.Dockerfile .

In [ ]:
# Test the environment
import sys
sys.path.insert(0, '.')

from sysadmin_env.environment import SysadminEnv
from sysadmin_env.models import Action

env = SysadminEnv()
obs = env.reset(scenario_id='nginx_syntax')
print(f"Scenario: nginx_syntax")
print(f"Complaint: {obs.output}")

obs2 = env.step(Action(command='systemctl status nginx'))
print(f"\nCommand output: {obs2.output[:300]}...")
print(f"Reward: {obs2.reward}")
env.close()
print("\nEnvironment works!")

## 2. Configuration

In [ ]:
# GRPO Training Configuration
CONFIG = {
    # Model - use 3B for T4, 7B for A100
    "model_name": "Qwen/Qwen2.5-Coder-3B-Instruct",
    
    # Start from SFT checkpoint if available
    "sft_checkpoint": None,  # Set to "checkpoints/sft" if you ran SFT first
    
    # LoRA settings
    "lora_r": 16,
    "lora_alpha": 16,
    
    # GRPO settings
    "num_steps": 100,           # Training steps
    "episodes_per_step": 8,     # Episodes to collect per step
    "group_size": 4,            # GRPO group size for advantage computation
    "max_turns": 10,            # Max commands per episode
    "learning_rate": 1e-5,
    
    # Generation
    "max_seq_length": 2048,
    "temperature": 0.7,
    
    # Paths
    "output_dir": "checkpoints/grpo",
    
    # Logging
    "use_wandb": False,  # Set True and login to enable
    "wandb_project": "sysadmin-game",
    "save_every": 25,
}

print("Configuration loaded!")
print(f"Model: {CONFIG['model_name']}")
print(f"Training steps: {CONFIG['num_steps']}")
print(f"Episodes per step: {CONFIG['episodes_per_step']}")

In [ ]:
# Optional: Login to W&B for experiment tracking
if CONFIG["use_wandb"]:
    import wandb
    wandb.login()
    wandb.init(project=CONFIG["wandb_project"], name="grpo-training", config=CONFIG)

## 3. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load from SFT checkpoint if available, otherwise base model
model_path = CONFIG["sft_checkpoint"] or CONFIG["model_name"]
print(f"Loading model: {model_path}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters if loading base model
if CONFIG["sft_checkpoint"] is None:
    print("Adding LoRA adapters to base model")
    model = FastLanguageModel.get_peft_model(
        model,
        r=CONFIG["lora_r"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

print(f"Model loaded! Device: {model.device}")

## 4. Define OpenEnv Interaction

In [ ]:
import re
from sysadmin_env.environment import SysadminEnv
from sysadmin_env.models import Action

# System prompt for the agent
SYSTEM_PROMPT = """You are an expert SRE agent that diagnoses and fixes Linux system issues.

When given a problem:
1. Think step-by-step about the diagnosis in <think> tags
2. Run ONE command at a time in <bash> tags
3. Analyze the output before running the next command
4. Continue until the issue is fixed

Example response:
<think>
The user reports nginx won't start. First, let me check the service status to see the error.
</think>
<bash>systemctl status nginx --no-pager -l</bash>

After seeing output, respond with your next diagnostic step or fix."""


def parse_response(response: str) -> tuple[str | None, str | None]:
    """Extract thinking and bash command from model response."""
    think_match = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    bash_match = re.search(r"<bash>(.*?)</bash>", response, re.DOTALL)
    
    thinking = think_match.group(1).strip() if think_match else None
    command = bash_match.group(1).strip() if bash_match else None
    
    return thinking, command


def run_episode(env, model, tokenizer, max_turns: int) -> dict:
    """Run a single episode with the OpenEnv environment.
    
    This is the core RL loop:
    1. Environment gives initial complaint
    2. Model generates diagnostic/fix command
    3. Environment executes command, returns output + reward
    4. Repeat until fixed or max turns
    """
    obs = env.reset()
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": obs.output},
    ]
    
    trajectory = {
        "scenario_id": obs.metadata["scenario_id"],
        "prompts": [],
        "responses": [],
        "rewards": [],
        "commands": [],
    }
    
    total_reward = 0.0
    
    for turn in range(max_turns):
        # Format prompt
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        
        # Generate response
        FastLanguageModel.for_inference(model)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=CONFIG["temperature"],
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:], 
            skip_special_tokens=True
        )
        
        # Parse response
        thinking, command = parse_response(response)
        
        # Store trajectory
        trajectory["prompts"].append(prompt)
        trajectory["responses"].append(response)
        
        # If no command, episode ends with penalty
        if not command:
            trajectory["rewards"].append(-0.1)
            trajectory["commands"].append(None)
            total_reward -= 0.1
            break
        
        trajectory["commands"].append(command)
        
        # Execute command in OpenEnv
        action = Action(command=command)
        obs = env.step(action)
        
        trajectory["rewards"].append(obs.reward)
        total_reward += obs.reward
        
        # Add to conversation
        messages.append({"role": "assistant", "content": response})
        messages.append({"role": "tool", "content": f"<output>\n{obs.output}\n</output>"})
        
        if obs.done:
            break
    
    trajectory["total_reward"] = total_reward
    trajectory["fixed"] = obs.metadata.get("fixed", False)
    trajectory["num_commands"] = len([c for c in trajectory["commands"] if c])
    
    return trajectory


print("OpenEnv interaction functions defined!")

In [ ]:
# Test a single episode
print("Testing single episode...")
print("="*50)

with SysadminEnv() as env:
    traj = run_episode(env, model, tokenizer, max_turns=5)

print(f"Scenario: {traj['scenario_id']}")
print(f"Commands executed: {traj['num_commands']}")
print(f"Total reward: {traj['total_reward']:.3f}")
print(f"Fixed: {traj['fixed']}")
print("\nCommands:")
for i, cmd in enumerate(traj['commands']):
    if cmd:
        print(f"  {i+1}. {cmd}")

## 5. GRPO Training Loop

In [ ]:
import os
from collections import defaultdict

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs("results", exist_ok=True)


def collect_trajectories(model, tokenizer, num_episodes: int) -> list[dict]:
    """Collect multiple episode trajectories from OpenEnv."""
    trajectories = []
    
    with SysadminEnv() as env:
        for i in range(num_episodes):
            try:
                traj = run_episode(env, model, tokenizer, CONFIG["max_turns"])
                trajectories.append(traj)
            except Exception as e:
                print(f"    Episode {i+1} failed: {e}")
                continue
    
    return trajectories


def compute_grpo_advantages(trajectories: list[dict], group_size: int) -> list[dict]:
    """Compute GRPO advantages: normalize rewards within groups."""
    if len(trajectories) < group_size:
        return []
    
    processed = []
    
    for i in range(0, len(trajectories) - group_size + 1, group_size):
        group = trajectories[i:i + group_size]
        rewards = [t["total_reward"] for t in group]
        
        # Compute mean and std for the group
        mean_reward = sum(rewards) / len(rewards)
        std_reward = (sum((r - mean_reward) ** 2 for r in rewards) / len(rewards)) ** 0.5
        std_reward = max(std_reward, 1e-6)
        
        # Normalize advantages within group
        for traj in group:
            advantage = (traj["total_reward"] - mean_reward) / std_reward
            
            # Create training examples
            for prompt, response, reward in zip(
                traj["prompts"], traj["responses"], traj["rewards"]
            ):
                processed.append({
                    "prompt": prompt,
                    "response": response,
                    "reward": reward,
                    "advantage": advantage,
                    "scenario_id": traj["scenario_id"],
                })
    
    return processed


print("GRPO functions defined!")

In [ ]:
# Training metrics storage
training_history = {
    "steps": [],
    "rewards": [],
    "fix_rates": [],
    "avg_commands": [],
    "losses": [],
}

# Per-scenario tracking
scenario_stats = defaultdict(lambda: {"attempts": 0, "fixes": 0})

print("Starting GRPO training...")
print(f"Steps: {CONFIG['num_steps']}")
print(f"Episodes per step: {CONFIG['episodes_per_step']}")
print("="*60)

In [ ]:
# Main training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

for step in range(CONFIG["num_steps"]):
    print(f"\nStep {step + 1}/{CONFIG['num_steps']}")
    
    # Collect trajectories from OpenEnv
    print(f"  Collecting {CONFIG['episodes_per_step']} episodes...")
    trajectories = collect_trajectories(
        model, tokenizer, 
        num_episodes=CONFIG["episodes_per_step"]
    )
    
    if not trajectories:
        print("  No successful trajectories, skipping step")
        continue
    
    # Compute metrics
    rewards = [t["total_reward"] for t in trajectories]
    fix_rate = sum(1 for t in trajectories if t["fixed"]) / len(trajectories)
    avg_reward = sum(rewards) / len(rewards)
    avg_commands = sum(t["num_commands"] for t in trajectories) / len(trajectories)
    
    # Update per-scenario stats
    for t in trajectories:
        scenario_stats[t["scenario_id"]]["attempts"] += 1
        if t["fixed"]:
            scenario_stats[t["scenario_id"]]["fixes"] += 1
    
    print(f"  Reward: {avg_reward:.3f} | Fix rate: {fix_rate:.1%} | Avg commands: {avg_commands:.1f}")
    
    # Compute GRPO advantages
    training_data = compute_grpo_advantages(trajectories, CONFIG["group_size"])
    
    if not training_data:
        continue
    
    # Policy gradient update
    FastLanguageModel.for_training(model)
    model.train()
    
    total_loss = 0.0
    num_updates = 0
    
    for example in training_data:
        inputs = tokenizer(
            example["prompt"] + example["response"],
            return_tensors="pt",
            truncation=True,
            max_length=CONFIG["max_seq_length"],
        ).to(model.device)
        
        outputs = model(**inputs, labels=inputs.input_ids)
        
        # Weight loss by advantage (GRPO objective)
        loss = outputs.loss * example["advantage"]
        
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        
        loss.backward()
        total_loss += loss.item()
        num_updates += 1
    
    # Gradient step
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad()
    
    avg_loss = total_loss / max(num_updates, 1)
    print(f"  Loss: {avg_loss:.4f}")
    
    # Store metrics
    training_history["steps"].append(step + 1)
    training_history["rewards"].append(avg_reward)
    training_history["fix_rates"].append(fix_rate)
    training_history["avg_commands"].append(avg_commands)
    training_history["losses"].append(avg_loss)
    
    # Log to W&B
    if CONFIG["use_wandb"]:
        wandb.log({
            "reward/mean": avg_reward,
            "reward/min": min(rewards),
            "reward/max": max(rewards),
            "success/fix_rate": fix_rate,
            "efficiency/avg_commands": avg_commands,
            "train/loss": avg_loss,
            "step": step + 1,
        })
    
    # Save checkpoint
    if (step + 1) % CONFIG["save_every"] == 0:
        checkpoint_path = f"{CONFIG['output_dir']}/step_{step + 1}"
        print(f"  Saving checkpoint to {checkpoint_path}")
        model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

print("\n" + "="*60)
print("GRPO training complete!")

In [ ]:
# Save final model
print(f"Saving final model to {CONFIG['output_dir']}")
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Model saved!")

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reward curve
ax = axes[0, 0]
ax.plot(training_history["steps"], training_history["rewards"], 'b-', linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Average Reward")
ax.set_title("GRPO Training - Reward Curve")
ax.grid(True, alpha=0.3)

# Fix rate curve
ax = axes[0, 1]
ax.plot(training_history["steps"], training_history["fix_rates"], 'g-', linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Fix Rate")
ax.set_title("GRPO Training - Success Rate")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Commands per episode
ax = axes[1, 0]
ax.plot(training_history["steps"], training_history["avg_commands"], 'orange', linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Average Commands")
ax.set_title("GRPO Training - Efficiency (Commands to Fix)")
ax.grid(True, alpha=0.3)

# Loss curve
ax = axes[1, 1]
ax.plot(training_history["steps"], training_history["losses"], 'r-', linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("GRPO Training - Policy Loss")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/grpo_training_curves.png", dpi=150)
plt.show()

print("Saved to results/grpo_training_curves.png")

In [ ]:
# Per-scenario performance
print("\nPer-Scenario Performance:")
print("-" * 50)
print(f"{'Scenario':<20} {'Attempts':>10} {'Fixes':>10} {'Rate':>10}")
print("-" * 50)

for scenario_id, stats in sorted(scenario_stats.items()):
    rate = stats["fixes"] / stats["attempts"] if stats["attempts"] > 0 else 0
    print(f"{scenario_id:<20} {stats['attempts']:>10} {stats['fixes']:>10} {rate:>10.1%}")

## 7. Test Trained Model

In [ ]:
# Run evaluation episodes
print("Evaluating trained model...")
print("="*60)

eval_results = []

with SysadminEnv() as env:
    for scenario_id in ["disk_full", "nginx_syntax", "port_bound", "ownership", "stale_pid"]:
        try:
            # Force specific scenario
            env.scenario_ids = [scenario_id]
            traj = run_episode(env, model, tokenizer, max_turns=10)
            
            eval_results.append({
                "scenario": scenario_id,
                "fixed": traj["fixed"],
                "commands": traj["num_commands"],
                "reward": traj["total_reward"],
            })
            
            status = "✅" if traj["fixed"] else "❌"
            print(f"{status} {scenario_id}: {traj['num_commands']} commands, reward={traj['total_reward']:.2f}")
            
        except Exception as e:
            print(f"❌ {scenario_id}: Failed - {e}")

print("\n" + "="*60)
fixed = sum(1 for r in eval_results if r["fixed"])
print(f"Evaluation: {fixed}/{len(eval_results)} scenarios fixed ({100*fixed/len(eval_results):.0f}%)")

In [ ]:
# Interactive test
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "nginx won't start, says something about a config error"}
]

input_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("--- Model Response ---")
print(response)

## 8. Compare with Baseline

In [ ]:
# If you have baseline results from SFT notebook, compare here
# This cell loads and compares metrics

import json
import os

# Save GRPO results
grpo_results = {
    "method": "GRPO",
    "final_reward": training_history["rewards"][-1] if training_history["rewards"] else 0,
    "final_fix_rate": training_history["fix_rates"][-1] if training_history["fix_rates"] else 0,
    "final_avg_commands": training_history["avg_commands"][-1] if training_history["avg_commands"] else 0,
    "eval_results": eval_results,
}

with open("results/grpo_results.json", "w") as f:
    json.dump(grpo_results, f, indent=2)

print("Results saved to results/grpo_results.json")
print("\nFinal Metrics:")
print(f"  Reward: {grpo_results['final_reward']:.3f}")
print(f"  Fix Rate: {grpo_results['final_fix_rate']:.1%}")
print(f"  Avg Commands: {grpo_results['final_avg_commands']:.1f}")

## 9. Download Results

In [ ]:
# Zip results for download
!zip -r grpo_results.zip checkpoints/grpo results/

print("\nDownload grpo_results.zip from the file browser (left panel)")

---

## Key Differences: SFT vs GRPO

| Aspect | SFT | GRPO |
|--------|-----|------|
| **Data** | Static traces (92 examples) | Live environment interaction |
| **Learning** | Imitation | Trial and error + reward |
| **Feedback** | None | Real fix/fail outcomes |
| **Exploration** | None | Agent discovers new strategies |
| **Compute** | ~30 min | ~5 hours |

**GRPO Advantage**: The model learns from actual outcomes, not just demonstrations. It can discover strategies not in the training data.

**SFT Advantage**: Faster, more stable, good for warm-starting before RL.

**Best Practice**: SFT warm-start → GRPO fine-tuning